# Philippines Model

In this section, we will explore a 3-region model of the Philippines (Mindanao, Visayas, Luzon) for the year 2025 that we are going to use to investigate a range of security of supply scenarios and metrics and how to consider uncertainty and risk in planning future energy systems. The model is adapted from the [Transition Zero Scenario Builder](https://www.transitionzero.org/products/scenario-builder). We start with

- three regions with demand and renewable generation profiles,
- existing power plant and storage capacities per region,
- existing inter-regional transmission capacities, and
- no capacity expansion (only dispatch).

## Input Inspection

In [ ]:
import pypsa
import pandas as pd

import plotly.io as pio
import plotly.offline as py

from pypsa.costs import annuity

pd.options.plotting.backend = "plotly"

Let's first load the network and inspect the data.

In [ ]:
url = "https://tubcloud.tu-berlin.de/s/M2QqWfES4Md8Xoe/download/phl-network.nc"
n = pypsa.Network(url)
n

Load profile (in MW) for the three regions:

In [ ]:
n.loads_t.p_set.plot()

Solar and wind capacity factors for the three regions:

In [ ]:
n.generators_t.p_max_pu.loc["2025-03"].filter(like="Visayas").plot()

Generators per region (aggregated by technology and region):

In [ ]:
n.generators.head(6)

Total installed capacity per technology and region:

In [ ]:
n.statistics.installed_capacity(groupby=["bus", "carrier"]).droplevel(
    "component"
).unstack("bus").plot.bar()

Regional distribtuion of electricty demand and generation capacity:

In [ ]:
load = n.loads_t.p_set.sum(axis=0).groupby(n.loads.bus).sum()
n.explore(bus_size=load / 1000, link_width=n.links.p_nom / 10)

In [ ]:
capacities = n.generators.groupby(["bus", "carrier"]).p_nom.sum()
n.explore(bus_size=capacities * 3, link_width=n.links.p_nom / 10)

## Dispatch Optimisation

Now, we can run a first simple dispatch optimisation to see how the system operates given these inputs; at this stage, we are not allowing any capacity expansion, so the model will only optimise the dispatch of existing assets to meet demand at minimum cost.

In [ ]:
n.optimize(log_to_console=False)

From the solved network, we can extract the operational expenditure (OPEX) per technology and as a total (in M$):

In [ ]:
opex = n.statistics.opex().div(1e6).round(1)
opex

In [ ]:
opex.sum()

And, of course, we can still inspect the energy balance time series:

In [ ]:
n.statistics.energy_balance.iplot()

On an interactive map, we might be interested in the electricity mix per region, the imbalance of supply and demand, or the inter-regional flows. 

In [ ]:
bus_size = (
    n.statistics.energy_balance(groupby=["bus", "carrier"])
    .droplevel("component")
    .drop("transmission", level="carrier")
)

link_loading = n.links_t.p0.mean().abs() / n.links.p_nom * 100

n.explore(
    bus_size=bus_size / 10000,
    bus_split_circle=True,
    link_color=link_loading,
    link_width=n.links.p_nom / 20,
)

We can also calculate the average cost of electricity supply by normalising the total cost (including CAPEX of existing assets) by the total demand:

In [ ]:
(n.statistics.capex().sum() + n.statistics.opex().sum()) / (
    n.snapshot_weightings.generators @ n.loads_t.p_set
).sum()

And the progression of regional marginal electricity prices in ($/MWh) over time:

In [ ]:
n.buses_t.marginal_price.plot(labels=dict(value="Electricity Price [$/MWh]"))

The total emissions of this system must be calculated, taking into account each generator's fuel type and emissions factor, as well as the amount of electricity generated:

In [ ]:
e = (
    n.generators_t.p.mul(n.snapshot_weightings.generators, axis=0)
    / n.generators.efficiency
    * n.generators.carrier.map(n.carriers.co2_emissions)
)
e.sum().sum()

In [ ]:
e.sum(axis=1).div(n.loads_t.p_set.sum(axis=1)).mul(1e3).plot()  # g/kWh

## Investment Optimisation

Now, suppose we want to investigate how a cost-effective system layout might look like if we were to introduce an **upper limit on total emissions**, and allow for capacity expansion of certain technologies.

Parameters to play with:

In [ ]:
EMISSION_FACTOR = 0.05
EXTENDABLE_GENERATORS = [
    "photovoltaic",
    "wind-onshore",
    "wind-offshore",
    "gas-ccs",
    "nuclear",
]
GREEN_FUEL_BACKUP = True
LOAD_SHEDDING = True

First, we add a global constraint to the model to limit total emissions relative to our original dispatch solution::

In [ ]:
n.add(
    "GlobalConstraint",
    "co2_limit",
    carrier_attribute="co2_emissions",
    sense="<=",
    constant=e.sum().sum() * EMISSION_FACTOR,
)

As the current system will unlikely be able to achieve this level of emissions reduction, we also need to allow for capacity expansion of certain technologies using the `p_nom_extendable` attribute.

In [ ]:
extendable = n.generators.carrier.isin(EXTENDABLE_GENERATORS)
n.generators["p_nom_extendable"] = extendable

extendable = n.storage_units.carrier == "battery-unspecified"
n.storage_units.loc[extendable, "p_nom_extendable"] = True

### Load Shedding

We can also add some demand elasticity to the model by allowing for load shedding at a certain cost; it represents the cost of unserved energy, and can be interpreted as the value of lost load (VoLL) or the cost of demand response measures. By setting it to a high value, we can ensure that load shedding only occurs when absolutely necessary to meet the emissions constraint.

In [ ]:
n.add(
    "Generator",
    n.buses.index + " load-shedding",
    bus=n.buses.index,
    carrier="load-shedding",
    lifetime=25,
    marginal_cost=2000,
    p_nom=n.loads_t.p_set.max().max(),
)
n.add("Carrier", "load-shedding", color="crimson")

### Backup Generation

In addition to the fossil fuel power plants, we can also add a generic *green fuel turbine* technology that can be used as backup generation when renewable generation is insufficient to meet demand, but with a limited amount of energy (e.g., 5 TWh using the `e_sum_max` attribute) to reflect the limited availability of sustainable fuels (e.g. hydrogen or a derivative). This technology would have a higher cost than fossil fuel generation, but zero emissions.

In [ ]:
n.add(
    "Generator",
    n.buses.index + " green-fuel-turbine",
    bus=n.buses.index,
    carrier="green-fuel-turbine",
    efficiency=0.4,
    lifetime=25,
    capital_cost=annuity(0.1, 25) * 1_000_000,
    marginal_cost=200,
    e_sum_max=5e6,  # 5 TWh
)

## Clustering

It is very common to use clustering techniques to reduce the temporal and spatial resolution of the model, which can significantly reduce the computational burden of the optimisation problem, especially when using free open-source solvers like HiGHS. With a commercial solver like Gurobi, models with much less reduction can be solved in reasonable time:

In [ ]:
# n.optimize(solver_name="gurobi", log_to_console=False)
# n.statistics.energy_balance.iplot()




Spatially, the model is already reduced to 3 regions, but we can further reduce the temporal resolution by clustering the time series data (e.g., demand and renewable generation profiles) into a smaller number of representative or resampled periods. This can be done using PyPSA built-in clustering methods. 



In [ ]:
n.model.solver_model = None
nc = n.cluster.temporal.segment(1000)
nc.loads_t.p_set.plot()

## Results

Now that we have reduced the temporal resolution, we can even solve the capacity expansion problem fairly quickly (at some loss of accuracy, of course) and investigate the results in a similar way as before, but now with the added dimension of spatial distribution of added generation, transmission and storage capacities. 

In [ ]:
nc.optimize(log_to_console=False)

Time series of energy balance:

In [ ]:
nc.statistics.energy_balance.iplot()

Time series of energy balance:

In [ ]:
n.statistics.energy_balance().div(1e6).round(1).sort_values()

Regional distribution of energy consumption and production by technology.

In [ ]:
bus_size = (
    nc.statistics.energy_balance(groupby=["bus", "carrier"])
    .droplevel("component")
    .drop("transmission", level="carrier")
)

link_loading = nc.links_t.p0.mean().abs() / nc.links.p_nom * 100

nc.explore(
    bus_size=bus_size / 10000,
    bus_split_circle=True,
    link_color=link_loading,
    link_width=nc.links.p_nom / 20,
)

Now, in addition to the OPEX, we can also look at the CAPEX of the system, and calculate the average cost of electricity supply as before:

In [ ]:
nc.statistics.capex().div(1e6).round(1)

In [ ]:
nc.statistics.opex().div(1e6).round(1)

In [ ]:
(nc.statistics.capex().sum() + nc.statistics.opex().sum()) / (
    nc.snapshot_weightings.generators @ nc.loads_t.p_set
).sum()

Finally, we can look at the shadow price of the emissions constraint, which gives us an indication of the marginal cost of reducing emissions in this system (i.e., the carbon price that would be needed to achieve this level of emissions reduction) and how the emission intensity was reduced over the year compared to the original dispatch solution:

In [ ]:
nc.global_constraints

In [ ]:
e = (
    nc.generators_t.p.mul(nc.snapshot_weightings.generators, axis=0)
    / nc.generators.efficiency
    * nc.generators.carrier.map(nc.carriers.co2_emissions)
)
load = nc.loads_t.p_set.sum(axis=1).mul(nc.snapshot_weightings.generators, axis=0)
e.sum(axis=1).div(load).mul(1e3).plot()  # g/kWh

## Excercises

**Task 1:** Investigate the impact of different levels of emissions reduction (e.g., 20%, 50%, 80%) on the system's cost, generation mix, and marginal prices. How does the system adapt to more stringent emissions constraints?

**Task 2:** Vary the parameters for the investment optimisation, such as the cost of load shedding or the availability of backup generation, and observe how these changes affect the optimal system configuration and operation.